<a href="https://colab.research.google.com/github/ritan612/S-AES-OFB/blob/main/IN410.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

## Initialization Vector(IV) And Input the Key

In [ ]:
import random

def initializing_IV():
  IV = random.getrandbits(16)
  # '016b' means : binary format (b), on 16 bits (16), padded with (0)
  print(f"Initialization Vector:{IV:016b}")
  return IV

def initializing_S_AES_key():
  while True:
      key = input("Enter 16-bit key: ")
      # Check length and allowed characters
      if len(key) == 16 and all(bit in '01' for bit in key):
          return key
      print("Invalid input! Please enter exactly 16 bits (only 0 and 1).")
key= initializing_S_AES_key()
# Convert to array (list of integers)
key_array = [int(bit) for bit in key]

print("Valid key:", key)
print("Key as array:", key_array)



Enter 16-bit key: 1111000011110000
Valid key: 1111000011110000
Key as array: [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0]


## Upload the file

In [ ]:
from google.colab import drive,files

drive.mount('/content/drive') # connects the notebook to google drive so we can read/write files

uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving Hello World.txt to Hello World (1).txt


## Convert the file into the Plaintext (bits array)

In [ ]:
def file_to_bits_array(file_path, chunk_size=8192):
    bits_array = []
    with open(file_path, "rb") as f: # "rb": read binary(reads raw bytes, every byte comes in as an integer 0-255)
        while chunk := f.read(chunk_size): # := (walrus operator assigns and checks in one step) read 8192 bytes and store them in chunk then check if chunk is not empty
            for byte in chunk:
                for i in range(7, -1, -1):# start from 7, count down (-1), stop befor -1 (stop on 0):these are the bit positions
                    bits_array.append((byte >> i) & 1) # ex: 01010110 for i=5 -> byte>>5 -> 01010 & 1 = 0
    return bits_array

bits = file_to_bits_array(uploaded_path)

print(bits[:64])  # first 64 bits
print(len(bits))   # total number of bits


[0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1]
96


## Split The Plaintext into 16-bit Blocks

In [ ]:
from itertools import batched

def batched_with_padding(iterable, batch_size, fill_value=0):
    for batch in batched(iterable, batch_size):
      """
           "yield" returns each batch when it's ready even if the for loop is not done. That's what makes it special it doesn't act like return (because "return" returns all the batches in one data structure what makes using "return" here inefficient for the memory)
      """
      yield batch + (fill_value,) * (batch_size - len(batch))


batches = {}
""" enumerate ask the "batched_with_padding" function for an iterable and since we used yield inside this function it will returns one batch and waits until
enumerate asks for one other batch etc...
"""
for i, batch in enumerate(batched_with_padding(bits, 16), start=1):
    name = f"p{i}"
    batches[name] = batch
    print(name, "=", batch)

def bits_to_int(bits):
    return int("".join(str(b) for b in bits), 2)

p1 = (0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1)
p2 = (0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0)
p3 = (0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0)
p4 = (0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1)
p5 = (0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0)
p6 = (0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0)


# Simplified AES (S-AES) implementation

## SBOX and Inverse SBOX

In [ ]:
SBOX = {0x0:0x9,0x1:0x4,0x2:0xA,0x3:0xB,
        0x4:0xD,0x5:0x1,0x6:0x8,0x7:0x5,
        0x8:0x6,0x9:0x2,0xA:0x0,0xB:0x3,
        0xC:0xC,0xD:0xE,0xE:0xF,0xF:0x7}

INV_SBOX = {v:k for k,v in SBOX.items()}


## Key Generation

In [ ]:
def sub_nib(b): return (SBOX[b>>4]<<4) | SBOX[b&0xF] #t2asem el bytes 2osmen w bet tabe2 3layoun el Sbox b>>4 awal nibble b&0xF ekhir nibble
def rot_nib(b): return ((b&0xF)<<4) | (b>>4) #tbadel el nosen

def key_expansion(key): #generate sub-key
  W0 = (key>>8)&0xFF # awal 8 w byedman enno 8 bits
  W1 = key&0xFF #ekhir 8 bits
  W2 = W0 ^ 0x80 ^ sub_nib(rot_nib(W1)) #RCon[1]=80 (10000000), ^ =XOR
  W3 = W2 ^ W1
  W4 = W2 ^ 0x30 ^ sub_nib(rot_nib(W3)) #RCon[2]=30 (00110000)
  W5 = W4 ^ W3
  K0 = (W0<<8)|W1
  K1 = (W2<<8)|W3
  K2 = (W4<<8)|W5
  print(f"Key0 = {K0:016b}")
  print(f"Key1 = {K1:016b}")
  print(f"Key2 = {K2:016b}")
  return K0,K1,K2


## Encryption

In [ ]:

def add_round_key(state,key): return state ^ key #Plaintext XOR Key0

def sub_nibbles(state): # n2asem kel nibble la 7al (4 nibbles)
    return (SBOX[(state>>12)&0xF]<<12 |
            SBOX[(state>>8)&0xF]<<8 |
            SBOX[(state>>4)&0xF]<<4 |
            SBOX[state&0xF])

def shift_rows(state):
    # swap 2nd and 4th nibble
    n0=(state>>12)&0xF
    n1=(state>>8)&0xF
    n2=(state>>4)&0xF
    n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def mix_columns(state):
    # simplified GF(2^4) multiplication
    def mult(a,b):
        res=0
        while b: #la tsir b = 0000
            if b&1: res^=a # res = b XOR a
            a<<=1 # shift a ex.: 0100 -> 1000 / 1000 -> 10000 overflow
            if a&0x10: a^=0x13 #x⁴ + x + 1  → 0x13 (10011) ex.: 10000 XOR 10011 = 00011
            b>>=1 # shift b ex.: 0011 -> 0001
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF # n2asem el nibbles
    n2=(state>>4)&0xF;  n3=state&0xF
    return ((n0 ^ mult(4,n1)) << 12 |
        (mult(4,n0) ^ n1) <<  8 |
        (n2 ^ mult(4,n3)) <<  4 |
        (mult(4,n2) ^ n3))

def encrypt(plaintext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Plaintext = {plaintext:016b}")
    state=add_round_key(plaintext,K0)
    print(f"After AddRoundKey0 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles1 = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows1   = {state:016b}")
    state=mix_columns(state)
    print(f"After MixColumns1  = {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1 = {state:016b}")
    state=sub_nibbles(state)
    print(f"After SubNibbles2  = {state:016b}")
    state=shift_rows(state)
    print(f"After ShiftRows2   = {state:016b}")
    state=add_round_key(state,K2)
    print(f"Ciphertext         = {state:016b}")
    return state


In [ ]:
# # Step 1: IV
# iv = initializing_IV()

# # make sure that key is an int
# if isinstance(key, str):
#     key = int(key, 2)

# cipher_blocks = {}
# O = iv

# for i, batch in enumerate(batched_with_padding(bits, 16), start=1):
#   name = f"p{i}"
#   batches[name] = batch
#   print(name, "=", batch)

#   # Step 2: Encrypt Using S-AES
#     O = encrypt(O, key)
#     print(f"O{i} = {O:016b}")

#     # Step 3: Convert Plaintext from bits to int
#     P = bits_to_int(batch)
#     print(f"{name} = {P:016b}")

#     # Step 4: XOR with plaintext
#     C = P ^ O
#     cipher_blocks[f"C{i}"] = C

#     print(f"C{i} = {C:016b}\n")



## Decryption

In [ ]:
def inv_sub_nibbles(state):
    return (INV_SBOX[(state>>12)&0xF]<<12 |
            INV_SBOX[(state>>8)&0xF]<<8 |
            INV_SBOX[(state>>4)&0xF]<<4 |
            INV_SBOX[state&0xF])

def inv_shift_rows(state):
    # same as shift_rows for 2nd and 4th nibble swap
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|(n1)

def inv_mix_columns(state):
    def mult(a,b):
        res=0
        while b:
            if b&1: res^=a
            a<<=1
            if a&0x10: a^=0x13
            b>>=1
        return res&0xF
    n0=(state>>12)&0xF; n1=(state>>8)&0xF
    n2=(state>>4)&0xF;  n3=state&0xF
    r0=mult(9,n0)^mult(2,n1)
    r1=mult(2,n0)^mult(9,n1)
    r2=mult(9,n2)^mult(2,n3)
    r3=mult(2,n2)^mult(9,n3)
    return (r0<<12)|(r1<<8)|(r2<<4)|r3

def decrypt(ciphertext,key):
    K0,K1,K2=key_expansion(key)
    print(f"Ciphertext = {ciphertext:016b}")
    state=add_round_key(ciphertext,K2)
    print(f"After AddRoundKey2 = {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows2 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles2= {state:016b}")
    state=add_round_key(state,K1)
    print(f"After AddRoundKey1  = {state:016b}")
    state=inv_mix_columns(state)
    print(f"After InvMixColumns1= {state:016b}")
    state=inv_shift_rows(state)
    print(f"After InvShiftRows1 = {state:016b}")
    state=inv_sub_nibbles(state)
    print(f"After InvSubNibbles1= {state:016b}")
    state=add_round_key(state,K0)
    print(f"Recovered Plaintext = {state:016b}")
    return state



# Implement OFB With S-AES

In [ ]:
# Step 1: IV
iv = initializing_IV()

# make sure that key is an int
if isinstance(key, str):
    key = int(key, 2)

cipher_blocks = {}
O = iv

for i, (name, bits_block) in enumerate(batches.items(), start=1):

    # Step 2: Encrypt Using S-AES
    O = encrypt(O, key)
    print(f"O{i} = {O:016b}")

    # Step 3: Convert Plaintext from bits to int
    P = bits_to_int(bits_block)
    print(f"{name} = {P:016b}")

    # Step 4: XOR with plaintext
    C = P ^ O
    cipher_blocks[f"C{i}"] = C

    print(f"C{i} = {C:016b}\n")

print("\nFinal Ciphertext:")
for name, value in cipher_blocks.items():
    print(f"{name} = {value:016b}")

binary_output = {
    "iv": format(iv, "016b"),
    "cipher": {
        name: format(value, "016b")
        for name, value in cipher_blocks.items()
    }
}

print(binary_output)

print("\n------Decryption--------")

iv = int(binary_output["iv"], 2)

cipher_blocks = {
    name: int(value, 2)
    for name, value in binary_output["cipher"].items()
}
decrypted_blocks = {}
O = iv

for i, (name, C) in enumerate(cipher_blocks.items(), start=1):
    O = encrypt(O, key)
    P = C ^ O # XOR is reversible
    decrypted_blocks[f"P{i}"] = P
    print(f"P{i} = {P:016b}\n")

Initialization Vector:0100011110100001
Key0 = 1111000011110000
Key1 = 1110011100010111
Key2 = 1000001110010100
Plaintext = 0100011110100001
After AddRoundKey0 = 1011011101010001
After SubNibbles1 = 0011010100010100
After ShiftRows1   = 0011010000010101
After MixColumns1  = 0000100001100001
After AddRoundKey1 = 1110111101110110
After SubNibbles2  = 1111011101011000
After ShiftRows2   = 1111100001010111
Ciphertext         = 0111101111000011
O1 = 0111101111000011
p1 = 0100100001100101
C1 = 0011001110100110

Key0 = 1111000011110000
Key1 = 1110011100010111
Key2 = 1000001110010100
Plaintext = 0111101111000011
After AddRoundKey0 = 1000101100110011
After SubNibbles1 = 0110001110111011
After ShiftRows1   = 0110101110110011
After MixColumns1  = 1100000001111001
After AddRoundKey1 = 0010011101101110
After SubNibbles2  = 1010010110001111
After ShiftRows2   = 1010111110000101
Ciphertext         = 0010110000010001
O2 = 0010110000010001
p2 = 0110110001101100
C2 = 0100000001111101

Key0 = 111100001111

# Brute Force Attack And Cryptanalysis

1

In [ ]:
# coding: utf-8
"""
  S-AES OFB — Brute Force & Known-Plaintext Attack
  Paste this AFTER the last cell of IN410.ipynb.
  Uses: binary_output, key, batches  (already defined above)

"""

import time

# Silent helpers (no print spam during key search)

def _sub_nib(b):    return (SBOX[b >> 4] << 4) | SBOX[b & 0xF]
def _rot_nib(b):    return ((b & 0xF) << 4) | (b >> 4)

def _key_exp(key):
    W0 = (key >> 8) & 0xFF;  W1 = key & 0xFF
    W2 = W0 ^ 0x80 ^ _sub_nib(_rot_nib(W1));  W3 = W2 ^ W1
    W4 = W2 ^ 0x30 ^ _sub_nib(_rot_nib(W3));  W5 = W4 ^ W3
    return (W0<<8)|W1, (W2<<8)|W3, (W4<<8)|W5

def _sub_nib_state(s):
    return (SBOX[(s>>12)&0xF]<<12 | SBOX[(s>>8)&0xF]<<8 |
            SBOX[(s>> 4)&0xF]<< 4 | SBOX[s&0xF])

def _shift_rows(s):
    n0=(s>>12)&0xF; n1=(s>>8)&0xF; n2=(s>>4)&0xF; n3=s&0xF
    return (n0<<12)|(n3<<8)|(n2<<4)|n1

def _mix_col(s):
    def m(a,b):
        r=0
        while b:
            if b&1: r^=a
            a<<=1
            if a&0x10: a^=0x13
            b>>=1
        return r&0xF
    n0=(s>>12)&0xF; n1=(s>>8)&0xF; n2=(s>>4)&0xF; n3=s&0xF
    return ((n0^m(4,n1))<<12|(m(4,n0)^n1)<<8|(n2^m(4,n3))<<4|(m(4,n2)^n3))

def _encrypt(pt, k):
    K0,K1,K2 = _key_exp(k)
    s = pt^K0
    s = _sub_nib_state(s); s = _shift_rows(s); s = _mix_col(s); s ^= K1
    s = _sub_nib_state(s); s = _shift_rows(s); s ^= K2
    return s

def _ofb_decrypt(iv_int, c_list, k):
    O = iv_int;  pt = []
    for C in c_list:
        O = _encrypt(O, k);  pt.append(C ^ O)
    return pt

def _to_text(blocks):
    raw = bytearray()
    for b in blocks: raw.append((b>>8)&0xFF); raw.append(b&0xFF)
    raw = raw.rstrip(b'\x00')
    try:    return raw.decode('utf-8')
    except: return raw.decode('latin-1')

def _printable(blocks):
    OK = set(range(0x20,0x7F))|{0x09,0x0A,0x0D,0x00}   # 0x00 = padding on last block
    for b in blocks:
        if ((b>>8)&0xFF) not in OK or (b&0xFF) not in OK: return False
    return True

# Pull the values your notebook already produced

_iv_int      = int(binary_output["iv"], 2)
_cipher_ints = [int(v, 2) for v in binary_output["cipher"].values()]
_key_int     = key if isinstance(key, int) else int(key, 2)

#  ATTACK 1 — BRUTE FORCE

print("=" * 60)
print("  ATTACK 1 — BRUTE FORCE")
print(f"  Key space : 2^16 = 65536 possible keys")
print(f"  IV        : {binary_output['iv']}")
print(f"  Blocks    : {len(_cipher_ints)}")
print("=" * 60)

_t0 = time.time()
_candidates = []

for _k in range(2**16):
    _pt = _ofb_decrypt(_iv_int, _cipher_ints, _k)
    if _printable(_pt):
        _candidates.append((_k, _pt))

print(f"\n  Tested  : 65536 keys in {time.time()-_t0:.2f} s")
print(f"  Found   : {len(_candidates)} candidate(s)\n")

for _k, _pt in _candidates:
    print(f"  ✓  Key      : {_k:016b}  (0x{_k:04X})")
    print(f"     Preview  : {_to_text(_pt)[:80]!r}")
    if _k == _key_int:
        print(f"     ★  Matches the real encryption key!")
    print()


#  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS
#  The attacker knows the first plaintext block P1 (e.g. file header / known text).
#  Here we take it directly from batches["p1"] to prove correctness.

_P1_known = bits_to_int(batches["p1"])   # attacker's known plaintext block

print("=" * 60)
print("  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS")
print(f"  Known P1  : {_P1_known:016b}")
print(f"  C1        : {_cipher_ints[0]:016b}")
_O1_target = _cipher_ints[0] ^ _P1_known
print(f"  Target O1 : {_O1_target:016b}  (= C1 XOR P1)")
print(f"  Goal      : find k  s.t.  Encrypt(IV, k) = O1")
print("=" * 60)

_t0 = time.time()
_found_key = None

for _k in range(2**16):
    if _encrypt(_iv_int, _k) == _O1_target:
        _found_key = _k
        break                     # unique solution — stop immediately

print(f"\n  Time elapsed : {time.time()-_t0:.4f} s")

if _found_key is None:
    print("  ✗  No key found — check the known plaintext value.")
else:
    print(f"  ✓  Key recovered : {_found_key:016b}  (0x{_found_key:04X})")
    if _found_key == _key_int:
        print(f"  ★  Matches the real encryption key!")

    _recovered = _ofb_decrypt(_iv_int, _cipher_ints, _found_key)
    print(f"\n  Decrypted blocks:")
    for i, (C, P) in enumerate(zip(_cipher_ints, _recovered), 1):
        print(f"    C{i} = {C:016b}  →  P{i} = {P:016b}")
    print(f"\n  Recovered text : {_to_text(_recovered)!r}")

# Final consistency check

print("\n" + "=" * 60)
print("  CONSISTENCY CHECK")
print("=" * 60)
_bf_keys = [k for k,_ in _candidates]
if _found_key is not None and _found_key in _bf_keys:
    print("  ✓  Both attacks agree on the same key.\n")
else:
    print("  ✗  Attacks disagree — review validator or known P1.\n")

  ATTACK 1 — BRUTE FORCE
  Key space : 2^16 = 65536 possible keys
  IV        : 0100011110100001
  Blocks    : 6

  Tested  : 65536 keys in 2.44 s
  Found   : 1 candidate(s)

  ✓  Key      : 1111000011110000  (0xF0F0)
     Preview  : 'Hello World.'
     ★  Matches the real encryption key!

  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS
  Known P1  : 0100100001100101
  C1        : 0011001110100110
  Target O1 : 0111101111000011  (= C1 XOR P1)
  Goal      : find k  s.t.  Encrypt(IV, k) = O1

  Time elapsed : 0.2095 s
  ✓  Key recovered : 1111000011110000  (0xF0F0)
  ★  Matches the real encryption key!

  Decrypted blocks:
    C1 = 0011001110100110  →  P1 = 0100100001100101
    C2 = 0100000001111101  →  P2 = 0110110001101100
    C3 = 0101011010001011  →  P3 = 0110111100100000
    C4 = 0010000010011111  →  P4 = 0101011101101111
    C5 = 1111100010001001  →  P5 = 0111001001101100
    C6 = 1001101010101110  →  P6 = 0110010000101110

  Recovered text : 'Hello World.'

  CONSISTENCY CHECK
  ✓  Bot

2

In [ ]:
# -*- coding: utf-8 -*-
"""
S-AES OFB – Brute Force Attack & Known-Plaintext Cryptanalysis

Compatible with IN410.ipynb (no changes to original code needed).
Paste this section AFTER your existing notebook cells, or run as a
standalone script by copying the S-AES helper functions below.
"""

import time

#  Re-use the exact S-AES primitives from your
#  notebook (copy-pasted, zero modifications)

SBOX = {0x0:0x9,0x1:0x4,0x2:0xA,0x3:0xB,
        0x4:0xD,0x5:0x1,0x6:0x8,0x7:0x5,
        0x8:0x6,0x9:0x2,0xA:0x0,0xB:0x3,
        0xC:0xC,0xD:0xE,0xE:0xF,0xF:0x7}

INV_SBOX = {v: k for k, v in SBOX.items()}



def key_expansion_silent(key):
    """Same logic as your key_expansion() but without print statements."""
    W0 = (key >> 8) & 0xFF
    W1 =  key       & 0xFF
    W2 = W0 ^ 0x80 ^ sub_nib(rot_nib(W1))
    W3 = W2 ^ W1
    W4 = W2 ^ 0x30 ^ sub_nib(rot_nib(W3))
    W5 = W4 ^ W3
    return (W0 << 8) | W1, (W2 << 8) | W3, (W4 << 8) | W5   # K0, K1, K2




def encrypt_silent(plaintext, key):
    """Exact encrypt() logic, no print statements – used inside attacks."""
    K0, K1, K2 = key_expansion_silent(key)
    state = plaintext ^ K0
    state = sub_nibbles(state)
    state = shift_rows(state)
    state = mix_columns(state)
    state ^= K1
    state = sub_nibbles(state)
    state = shift_rows(state)
    state ^= K2
    return state

def ofb_decrypt_silent(iv, cipher_blocks_list, key):
    """
    OFB decryption for a list of integer cipher blocks.
    Returns a list of integer plaintext blocks.
    """
    O = iv
    plaintext_blocks = []
    for C in cipher_blocks_list:
        O = encrypt_silent(O, key)   # OFB: next keystream word
        plaintext_blocks.append(C ^ O)
    return plaintext_blocks

#  Validator helpers
#  OFB brute-force needs a way to decide "does this decryption look correct?"
#  Two heuristics are provided; you can combine them.

def is_printable_ascii(blocks):
    """
    Returns True if every recovered byte falls in the printable-ASCII range
    (0x20-0x7E) or is a common control char (tab, newline, CR).
    Good for .txt files.
    """
    ALLOWED = set(range(0x20, 0x7F)) | {0x09, 0x0A, 0x0D}
    for block in blocks:
        high = (block >> 8) & 0xFF
        low  =  block       & 0xFF
        if high not in ALLOWED or low not in ALLOWED:
            return False
    return True

def has_file_magic(blocks, magic_bytes):
    """
    Returns True if the recovered plaintext starts with a known file signature.
    Pass magic_bytes as a list of ints, e.g. [0xFF, 0xD8] for JPEG.

    Common signatures:
        PDF  : [0x25, 0x50, 0x44, 0x46]  → %PDF
        JPEG : [0xFF, 0xD8, 0xFF]
        PNG  : [0x89, 0x50, 0x4E, 0x47]
        ZIP  : [0x50, 0x4B, 0x03, 0x04]
        TXT  : use is_printable_ascii() instead
    """
    # Reassemble bytes from 16-bit blocks
    recovered_bytes = []
    for b in blocks:
        recovered_bytes.append((b >> 8) & 0xFF)
        recovered_bytes.append( b       & 0xFF)
    return recovered_bytes[:len(magic_bytes)] == magic_bytes



#  ATTACK 1 – BRUTE FORCE
#  Try every possible 16-bit key (0x0000 … 0xFFFF) and keep candidates
#  whose decrypted output passes the validator.

def brute_force_attack(binary_output, validator=is_printable_ascii):
    """
    Parameters
    ----------
    binary_output : dict  – exactly the dict your notebook produces:
        {
          "iv":     "0101...0101",          # 16-bit binary string
          "cipher": { "C1": "...", ... }     # binary strings
        }
    validator     : callable(list[int]) -> bool
        Function that receives recovered plaintext blocks (list of ints)
        and returns True if the decryption looks legitimate.

    Returns
    -------
    candidates : list of (key_int, plaintext_blocks)
    """
    iv            = int(binary_output["iv"], 2)
    cipher_ints   = [int(v, 2) for v in binary_output["cipher"].values()]
    key_space     = 2 ** 16          # 65 536 keys
    candidates    = []

    print("=" * 60)
    print("  ATTACK 1 — BRUTE FORCE")
    print(f"  Key space : 2^16 = {key_space} keys")
    print(f"  IV        : {iv:016b}")
    print(f"  Blocks    : {len(cipher_ints)}")
    print("=" * 60)

    t0 = time.time()

    for k in range(key_space):
        pt_blocks = ofb_decrypt_silent(iv, cipher_ints, k)
        if validator(pt_blocks):
            candidates.append((k, pt_blocks))

    elapsed = time.time() - t0

    print(f"\n  Tested  : {key_space} keys in {elapsed:.2f} s")
    print(f"  Found   : {len(candidates)} candidate(s)\n")

    for key_int, pt_blocks in candidates:
        print(f"  ✓  Candidate key : {key_int:016b}  (0x{key_int:04X})")
        # Show first recovered block
        print(f"     First block   : {pt_blocks[0]:016b}")
        # Try to show as text
        chars = _blocks_to_text(pt_blocks)
        preview = chars[:80].replace('\n', '↵').replace('\r', '')
        print(f"     Text preview  : {preview!r}")
        print()

    return candidates


#  ATTACK 2 – KNOWN-PLAINTEXT CRYPTANALYSIS
#
#  Theory:
#  In OFB mode:
#      O_1 = Encrypt(IV,  key)
#      C_1 = P_1 XOR O_1
#
#  If the attacker knows P_1 (first plaintext block), they can compute:
#      O_1 = C_1 XOR P_1     (known keystream output)
#
#  Then search for the key k* such that:
#      Encrypt(IV, k*) = O_1
#
#  This is still a 2^16 search, but it is DETERMINISTIC (no heuristic
#  needed) – exactly ONE key will satisfy the equation.
#
#  Once k* is found, decrypt ALL blocks with that key.

def known_plaintext_attack(binary_output, known_plaintext_block):
    """
    Parameters:

    binary_output         : dict – same structure as above
    known_plaintext_block : int  – the integer value of the first plaintext
                                   block (P_1) that the attacker knows.
                                   E.g. if the file starts with "He" → 0x4865

    Returns:

    (found_key, plaintext_blocks) or (None, []) if not found
    """
    iv          = int(binary_output["iv"], 2)
    cipher_ints = [int(v, 2) for v in binary_output["cipher"].values()]
    C1          = cipher_ints[0]
    key_space   = 2 ** 16

    # Step 1 – recover the first keystream output word
    O1 = C1 ^ known_plaintext_block

    print("=" * 60)
    print("  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS")
    print(f"  IV              : {iv:016b}")
    print(f"  Known P_1       : {known_plaintext_block:016b}")
    print(f"  C_1             : {C1:016b}")
    print(f"  Target O_1      : {O1:016b}  (= C_1 XOR P_1)")
    print(f"  Goal            : find k  s.t.  Encrypt(IV, k) = O_1")
    print("=" * 60)

    t0         = time.time()
    found_key  = None

    for k in range(key_space):
        if encrypt_silent(iv, k) == O1:
            found_key = k
            break                   # Only one key can satisfy this

    elapsed = time.time() - t0

    if found_key is None:
        print(f"\n  ✗  No key found in {elapsed:.2f} s – check P_1 value.\n")
        return None, []

    print(f"\n  ✓  Key recovered : {found_key:016b}  (0x{found_key:04X})")
    print(f"  Time elapsed    : {elapsed:.4f} s")

    # Step 2 – decrypt all blocks with the recovered key
    pt_blocks = ofb_decrypt_silent(iv, cipher_ints, found_key)

    print(f"\n  Decrypted {len(pt_blocks)} block(s):")
    for i, (C, P) in enumerate(zip(cipher_ints, pt_blocks), 1):
        print(f"    C{i} = {C:016b}  →  P{i} = {P:016b}")

    chars = _blocks_to_text(pt_blocks)
    print(f"\n  Recovered text (first 200 chars):\n  {chars[:200]!r}\n")

    return found_key, pt_blocks


#  Utility – convert recovered 16-bit blocks back to a readable string

def _blocks_to_text(blocks):
    raw = bytearray()
    for b in blocks:
        raw.append((b >> 8) & 0xFF)
        raw.append( b       & 0xFF)
    # Remove trailing zero-padding that was added before encryption
    raw = raw.rstrip(b'\x00')
    try:
        return raw.decode('utf-8')
    except UnicodeDecodeError:
        return raw.decode('latin-1')


def blocks_to_file(blocks, output_path):
    """Save recovered plaintext blocks back to a file (binary-safe)."""
    raw = bytearray()
    for b in blocks:
        raw.append((b >> 8) & 0xFF)
        raw.append( b       & 0xFF)
    raw = raw.rstrip(b'\x00')
    with open(output_path, 'wb') as f:
        f.write(raw)
    print(f"  Saved to: {output_path}")


#  DEMO  –  run both attacks against a self-generated test vector
#  Replace 'binary_output' with the actual dict produced by your notebook.

if __name__ == "__main__":

    # Simulate what your notebook already produces
    SECRET_KEY   = 0b0000000000000000   # pretend this is unknown to attacker
    PLAINTEXT    = "Hello S-AES OFB!"   # 16 chars = 16 bytes = 8 blocks of 16 bits

    import random
    random.seed(42)
    sim_iv = random.getrandbits(16)

    # Build plaintext blocks
    raw_bytes = PLAINTEXT.encode('utf-8')
    # Pad to even length
    if len(raw_bytes) % 2:
        raw_bytes += b'\x00'
    pt_ints = []
    for i in range(0, len(raw_bytes), 2):
        pt_ints.append((raw_bytes[i] << 8) | raw_bytes[i+1])

    # Encrypt with OFB
    O = sim_iv
    cipher_ints = []
    for P in pt_ints:
        O = encrypt_silent(O, SECRET_KEY)
        cipher_ints.append(P ^ O)

    # Package into binary_output (the format your notebook outputs)



    print("\n" + "━"*60)
    print("  DEMO: attacking the following ciphertext")
    print(f"  IV         : {binary_output['iv']}")
    print(f"  Plaintext  : {PLAINTEXT!r}  (secret)")
    print(f"  Secret key : {SECRET_KEY:016b}  (secret)")
    print("━"*60 + "\n")

    # ATTACK 1: Brute Force (text file heuristic)
    candidates = brute_force_attack(binary_output, validator=is_printable_ascii)

    # ATTACK 2: Known-Plaintext (attacker knows P_1 = "He" = 0x4865)
    known_P1 = (ord('H') << 8) | ord('e')   # 0x4865
    found_key, recovered_blocks = known_plaintext_attack(binary_output, known_P1)

    # Verify both attacks agree
    if candidates and found_key is not None:
        bf_keys = [k for k, _ in candidates]
        if found_key in bf_keys:
            print("  ✓  Both attacks recovered the SAME key – consistent!\n")
        else:
            print("  ✗  Mismatch between attacks – check validator logic.\n")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DEMO: attacking the following ciphertext
  IV         : 0100011110100001
  Plaintext  : 'Hello S-AES OFB!'  (secret)
  Secret key : 0000000000000000  (secret)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ATTACK 1 — BRUTE FORCE
  Key space : 2^16 = 65536 keys
  IV        : 0100011110100001
  Blocks    : 6

  Tested  : 65536 keys in 1.58 s
  Found   : 1 candidate(s)

  ✓  Candidate key : 1111000011110000  (0xF0F0)
     First block   : 0100100001100101
     Text preview  : 'Hello World.'

  ATTACK 2 — KNOWN-PLAINTEXT CRYPTANALYSIS
  IV              : 0100011110100001
  Known P_1       : 0100100001100101
  C_1             : 0011001110100110
  Target O_1      : 0111101111000011  (= C_1 XOR P_1)
  Goal            : find k  s.t.  Encrypt(IV, k) = O_1

  ✓  Key recovered : 1111000011110000  (0xF0F0)
  Time elapsed    : 0.2049 s

  Decrypted 6 block(s):
    C1 = 0011001110100110  →  P1 = 0100100001100101
    C2 =